# Stg Layer — comments (every read, insert-only)

Read **comments** from Bronze (`${bronze}.comments_raw`, append-only copy of Lakebase Postgres) → Silver `${silver}.stg_comments`.

**Grain (since 2026-11-02):** 1 row per `comment_id` × **`snapshot_ts`** (= `COALESCE(updated_at, created_at)`, when the app read the values from YouTube). Every read is kept, so the 07 / 09 / 21h stats refreshes no longer overwrite each other (before: 1 row per comment_id per day, the latest read of the day won). Nothing is updated: insert-only MERGE on (`comment_id`, `snapshot_ts`). `snapshot_date` = the Vietnam day of `snapshot_ts`.

Comments are fetched again 30 days later (`fetch_comments`), so `like_count` / `total_reply_count` now keep their history. Gold keeps only the latest read in `gold.dim_comment` (SCD 1; `fact_comment` / `fact_comment_snapshot` were dropped on 2026-11-02 because comment likes are not tracked), so this table is the only place with every read.

**Current values:** view `${silver}.stg_comments_latest` (latest read per `comment_id`, same columns).

**Flow:** `%run ./_history` → `history_load`: watermark `MAX(snapshot_ts)`, new Bronze rows, one row per grain, insert-only MERGE, refresh the `_latest` view, grain check. The first run after 2026-11-02 renames the old table to `stg_comments_bak_20261102` and rebuilds from Bronze + that table.

In [ ]:
%run ./_history

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE_TABLE = f"{BRONZE}.comments_raw"  # Bronze: append-only copy of every Postgres read
DELTA_TABLE = f"{SILVER}.stg_comments"
print(f"Bronze source: {SOURCE_TABLE} · Silver target: {DELTA_TABLE} (+ view {DELTA_TABLE}_latest)")

In [ ]:
DDL = """
    comment_id          STRING,
    snapshot_ts         TIMESTAMP,
    snapshot_date       DATE,
    video_id            STRING,
    author_display_name STRING,
    author_channel_id   STRING,
    text                STRING,
    like_count          BIGINT,
    published_at        TIMESTAMP,
    total_reply_count   INT,
    created_at          TIMESTAMP,
    updated_at          TIMESTAMP,
    source              STRING,
    etl_date            TIMESTAMP,
    op_type             STRING
"""
added = history_load(SOURCE_TABLE, DELTA_TABLE, "comment_id", DDL)

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT comment_id) AS ids, COUNT(DISTINCT snapshot_date) AS days, MAX(snapshot_ts) AS last_read
    FROM {DELTA_TABLE}
""").collect()[0]
print(f"📊 {DELTA_TABLE}: {s['n']} reads of {s['ids']} comment_ids over {s['days']} day(s), last read {s['last_read']}, added this run: {added}")
display(spark.sql(f"SELECT snapshot_date, COUNT(*) AS reads, COUNT(DISTINCT comment_id) AS ids, COUNT(DISTINCT snapshot_ts) AS read_times FROM {DELTA_TABLE} GROUP BY 1 ORDER BY 1 DESC LIMIT 14"))
print("\n--- stg_comments ETL complete ---")